# 07 — Tuning the logistic model's display shrinkage

The logistic model (`momentum_logistic_rolling`) displays `0.5 + display_shrinkage * (p - 0.5)`,
where `p` is its walk-forward logistic estimate. This notebook sweeps `display_shrinkage` to find
the value that balances two opposing forces, on the common out-of-sample window:

- **House edge on uninformed (noise) flow** — the vig the house keeps from the dominant flow.
- **Informed attacker edge** — a flatter display (low shrinkage) hands edge back to attackers
  who know the true `p`; a sharper display (shrinkage near 1.0) prices them out but can be
  over-confident on the noise.

The lead-lag attacker is independent of this knob (the logistic does not use the fast feed), so
only the **predictive** and **regime-aware** attackers are swept here.

In [ ]:
import os
import sys

# Move up one level to the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
# Add the new current directory to the Python path
sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np
import pandas as pd

from snapmarket.data import load_oracle_prices
from snapmarket.features import build_features
from snapmarket.parameters import SharedParameters
from snapmarket.models import build_model
from snapmarket.models.momentum_logistic_rolling import MomentumLogisticRollingParameters
from snapmarket.signals import walk_forward_logistic_probability, regime_conditional_probability
from snapmarket.strategies import noise_pool, predictive_bettor, regime_aware_bettor
from snapmarket.engine import simulate

shared_parameters = SharedParameters()
features = build_features(load_oracle_prices(), shared_parameters)

logistic_probability = walk_forward_logistic_probability(features, shared_parameters)
regime_probability = regime_conditional_probability(features, shared_parameters)
attackers = {
    "predictive": predictive_bettor(logistic_probability),
    "regime_aware": regime_aware_bettor(regime_probability),
}
print("ready")

## Sweep

For each shrinkage we rebuild the model, then over the same out-of-sample windows we measure the
uninformed house edge and each attacker's edge and volume share (all attackers run inside a noise
pool, same seeds across shrinkage values).

In [ ]:
shrinkage_values = [0.40, 0.55, 0.70, 0.85, 1.00]
window_length = 150_000
number_of_windows = 6


def aggregate_edge(model, bettors, seed_base):
    pnl = stake = volume = 0.0
    start0 = model.first_evaluation_index
    for window_index in range(number_of_windows):
        start = start0 + window_index * window_length
        if start + window_length + shared_parameters.horizon_seconds >= features.number_of_seconds:
            break
        result = simulate(model, features, bettors, start, window_length, seed=seed_base + window_index)
        volume += result.total_volume
        if "attacker" in result.per_bettor:
            outcome = result.per_bettor["attacker"]
            pnl += outcome.pnl
            stake += outcome.stake
        else:
            pnl += result.house_pnl
            stake += result.total_volume
    return pnl / stake if stake else 0.0, (stake / volume if volume else 0.0)


rows = []
for shrinkage in shrinkage_values:
    model = build_model("momentum_logistic_rolling", features, shared_parameters,
                        MomentumLogisticRollingParameters(display_shrinkage=shrinkage))
    house_edge, _ = aggregate_edge(model, {"noise": noise_pool()}, seed_base=1_000)
    row = {"shrinkage": shrinkage, "house_edge": house_edge}
    for name, bettor in attackers.items():
        edge, volume_share = aggregate_edge(model, {"pool": noise_pool(), "attacker": bettor}, seed_base=200)
        row[f"{name}_edge"] = edge
        row[f"{name}_volume_share"] = volume_share
    rows.append(row)

sweep = pd.DataFrame(rows).set_index("shrinkage")
sweep

In [ ]:
percent_columns = ["house_edge", "predictive_edge", "predictive_volume_share",
                   "regime_aware_edge", "regime_aware_volume_share"]
print(sweep[percent_columns].mul(100).round(3).to_string())

sweep["worst_attacker_edge"] = sweep[["predictive_edge", "regime_aware_edge"]].max(axis=1)
safe = sweep[sweep["worst_attacker_edge"] <= 0.0]
if len(safe):
    best = safe["house_edge"].idxmax()
    print(f"\nHighest house edge with no profitable attacker: shrinkage = {best:.2f} "
          f"(house edge {safe.loc[best, 'house_edge']:+.3%})")
else:
    least_leak = sweep["worst_attacker_edge"].idxmin()
    print(f"\nNo shrinkage fully prices out every attacker; least leakage at shrinkage = {least_leak:.2f} "
          f"(worst attacker edge {sweep.loc[least_leak, 'worst_attacker_edge']:+.3%})")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(sweep.index, sweep["house_edge"], "o-", color="#16a34a")
axes[0].axhline(shared_parameters.house_margin, ls="--", color="#9ca3af",
                label=f"vig ({shared_parameters.house_margin:.1%})")
axes[0].set_title("Uninformed house edge vs display shrinkage")
axes[0].set_xlabel("display shrinkage"); axes[0].set_ylabel("house edge")
axes[0].legend(fontsize=8)

axes[1].plot(sweep.index, sweep["predictive_edge"], "o-", color="#2563eb", label="predictive")
axes[1].plot(sweep.index, sweep["regime_aware_edge"], "o-", color="#dc2626", label="regime-aware")
axes[1].axhline(0, ls="--", color="#16a34a", label="break-even")
axes[1].set_title("Informed attacker edge vs display shrinkage")
axes[1].set_xlabel("display shrinkage"); axes[1].set_ylabel("attacker edge")
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

## Recorded results

Run on the common out-of-sample window, 6 windows of 150,000 seconds, same seeds across shrinkage values.

| display shrinkage | house edge (uninformed) | predictive edge (volume share) | regime-aware edge (volume share) |
|---|---|---|---|
| 0.40 | +12.90% | -1.80% (2.9%) | +2.15% (20.7%) |
| 0.55 | +12.96% | -14.84% (0.7%) | +0.88% (19.5%) |
| 0.70 | **+12.98%** | -52.84% (0.02%) | **-0.32% (18.5%)** |
| 0.85 | +12.94% | 0.00% (0%) | -1.08% (17.6%) |
| 1.00 | +12.85% | 0.00% (0%) | -1.68% (17.1%) |

**Conclusion: `display_shrinkage` close to 0.70.** That is where the uninformed house edge peaks (+12.98%) and where the regime-aware attacker first stops being profitable (-0.32%). Above 0.70 the house gives back a little vig for no extra protection; below 0.70 the regime-aware attacker turns profitable. The predictive attacker's large negative numbers in the middle are a tiny-volume artifact: it only bets in rare extreme-confidence cells (its volume share is near zero there).

![display shrinkage sweep](assets/07_display_shrinkage_sweep.png)